# PostHarvest IQ — Production ML Notebook
**WFP Code4FoodSecurity Capstone · Blossom Academy 2026**

End-to-end ML pipeline: price forecasting + sell/store decision support for Ghanaian smallholder farmers.

| Section | What it does |
|---------|-------------|
| 1. Setup | Install packages, imports, seeds, constants |
| 2. Data loading | Upload `cereals_merged_clean.csv` |
| 3. Feature engineering | Lags, rolling stats, seasonal flags, YoY change |
| 4. Decision labels | STORE / SELL_PARTIAL / SELL_NOW from `recommendation_service.py` logic |
| 5. Preprocessing | `ColumnTransformer` + `OneHotEncoder` pipeline fitted on train only |
| 6. Forecasting | Multivariate LSTM + early stopping + walk-forward validation + baselines |
| 7. Classification | Class weight vs SMOTE comparison + constrained hyperparameter tuning |
| 8. Explainability | SHAP values per decision class |
| 9. Evaluation | Full metrics, confusion matrix, comparison table |
| 10. Conclusions | Business insights, limitations, next steps |
| 11. Save artifacts | All models, scalers, pipeline, metadata |
| 12. Download | Download everything for VS Code |


## Section 1 — Setup

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib torch imbalanced-learn shap --quiet
print("All packages installed ")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json, os, random, warnings, shutil
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              classification_report, accuracy_score, confusion_matrix, f1_score)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from scipy.stats import randint, uniform
import xgboost as xgb
import joblib, shap

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("All libraries imported ")
print(f"PyTorch : {torch.__version__}  |  GPU : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {DEVICE}")

In [ ]:
#  Reproducibility — fix all random seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

#  Split ratios (chronological — no shuffle)
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

#  LSTM architecture
SEQ_LEN      = 12     # months of history  predict month 13
LSTM_HIDDEN  = 64
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.2
LSTM_EPOCHS  = 100
LSTM_BATCH   = 32
LSTM_LR      = 0.001
PATIENCE     = 15     # early stopping

#  Business logic — mirrors recommendation_service.py EXACTLY
STORAGE_COST = 5.0    # GHS/bag/month
STORAGE_MO   = 1.5    # months
TRANSPORT    = 2.0    # GHS/bag
STORE_THR    = 20.0   # net > 20 → STORE
PARTIAL_THR  = 5.0    # net > 5  → SELL_PARTIAL

# Output folder
MODELS_DIR = 'models'
CHECKPOINT = f'{MODELS_DIR}/lstm_best.pt'
os.makedirs(MODELS_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
PALETTE = {'STORE':'#27ae60','SELL_PARTIAL':'#e67e22','SELL_NOW':'#e74c3c'}
print("Configuration ready ")
print(f"Split: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}  |  LSTM seq={SEQ_LEN}  |  patience={PATIENCE}")

## Section 2 — Data Loading

Upload `cereals_merged_clean.csv`


In [ ]:
from google.colab import files

print("Upload cereals_merged_clean.csv ...")
uploaded = files.upload()
fname    = list(uploaded.keys())[0]

df_raw = pd.read_csv(fname)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)

print(f"Shape      : {df_raw.shape}")
print(f"Columns    : {list(df_raw.columns)}")
print(f"Markets    : {sorted(df_raw['market'].unique())}")
print(f"Commodities: {sorted(df_raw['commodity'].unique())}")
print(f"Date range : {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
print(f"Nulls      : {df_raw.isnull().sum().sum()}")
print()
print(df_raw.head(4).to_string())

## Section 3 — Feature Engineering

**Lag features** give the model memory of recent prices.
Lags are computed per market-commodity group so Tamale-Maize history
does not bleed into Bolga-Sorghum.

**Seasonal flags** tell the model whether we are in harvest season
(Oct–Dec, prices fall) or lean season (Jun–Aug, prices rise).
These are the two most reliable signals for the STORE decision.

**YoY price change** captures multi-year trend — important because
Ghana cereal prices trended sharply upward 2020–2023 due to currency
depreciation and global supply shocks.

**price_vs_annual** shows how current price compares to its own 12-month
average — a value above 1.0 means prices are unusually high right now.


In [ ]:
df = df_raw.copy()
df = df.sort_values(['market','commodity','date']).reset_index(drop=True)

grp = df.groupby(['market','commodity'])['price']

# Lag and rolling features
df['price_lag1']       = grp.shift(1)
df['price_lag2']       = grp.shift(2)
df['price_lag3']       = grp.shift(3)
df['rolling_mean_3']   = grp.transform(lambda x: x.rolling(3, min_periods=1).mean())
df['rolling_std_3']    = grp.transform(lambda x: x.rolling(3, min_periods=2).std())
df['price_pct_change'] = grp.pct_change()
df['price_next']       = grp.shift(-1)

#  Seasonal and trend features — key fix for minority class detection
df['month']             = df['date'].dt.month
df['is_harvest_season'] = df['month'].isin([10, 11, 12]).astype(int)
df['is_lean_season']    = df['month'].isin([6, 7, 8]).astype(int)
df['price_vs_annual']   = df['price'] / grp.transform(
    lambda x: x.rolling(12, min_periods=6).mean())
df['price_yoy']         = grp.pct_change(12)

df = df.dropna(subset=['price_lag1','price_lag2','price_lag3','rolling_std_3',
                        'price_pct_change','price_next','price_vs_annual','price_yoy'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Rows after feature engineering : {len(df)}")
print(f"Date range preserved           : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"New features: is_harvest_season, is_lean_season, price_vs_annual, price_yoy")

## Section 4 — Decision Labels

The label is derived from the exact formula in `recommendation_service.py`:

```
net_per_bag = (price_next − price_now) − (storage_cost × 1.5 months) − transport
STORE        if net > 20
SELL_PARTIAL if net > 5
SELL_NOW     otherwise
```

**Important:** 83% of records are SELL_NOW. This is not a data error — it
reflects real market economics. The median net per bag is GHS −9.25, meaning
storing loses money more than half the time in this dataset. A model that
mostly predicts SELL_NOW is correct. The goal is to correctly identify the
15% of months where storing or partial selling IS worth it.


In [ ]:
df['net_per_bag'] = (
    (df['price_next'] - df['price'])
    - (STORAGE_COST * STORAGE_MO)
    - TRANSPORT
)

def make_label(net):
    if net > STORE_THR:    return 'STORE'
    elif net > PARTIAL_THR: return 'SELL_PARTIAL'
    else:                   return 'SELL_NOW'

df['decision'] = df['net_per_bag'].apply(make_label)

counts = df['decision'].value_counts()
total  = len(df)
print("Net per bag stats:")
print(df['net_per_bag'].describe().round(2))
print()
print("Class distribution:")
for lbl, cnt in counts.items():
    bar = '█' * int(cnt/total*35)
    print(f"  {lbl:<15}: {cnt:>4} ({cnt/total*100:5.1f}%)  {bar}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Decision Label Distribution', fontsize=13, fontweight='bold')
bc = [PALETTE.get(d,'steelblue') for d in counts.index]
axes[0].bar(counts.index, counts.values, color=bc, edgecolor='white')
axes[0].set_title('Count per class'); axes[0].set_ylabel('Count')
for i,(l,c) in enumerate(counts.items()): axes[0].text(i, c+1, str(c), ha='center')
axes[1].pie(counts.values, labels=counts.index, colors=bc, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class share')
plt.tight_layout(); plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

## Section 5 — Preprocessing Pipeline

### Why `ColumnTransformer` + `OneHotEncoder` instead of `pd.get_dummies`

`pd.get_dummies` is fragile in production: if a market is missing from a
new batch of data the column disappears and inference crashes.

`sklearn` `OneHotEncoder` with `handle_unknown='ignore'` silently zeros out
unseen categories. The fitted pipeline is saved as an artefact so inference
at API time uses the exact same encoding the model was trained on.

The pipeline is **fitted on training data only** — test data is transformed
using training statistics. This prevents any leakage of test-set categories
into the encoding.

### Why `LabelEncoder` on the target label (y)

`market` and `commodity` (input features) use `OneHotEncoder`.
The decision label (what we predict) uses `LabelEncoder` to convert
SELL_NOW / SELL_PARTIAL / STORE to integers 0/1/2.
XGBoost requires integer class labels — string labels raise `ValueError`.
This is correct and standard — do not confuse encoding inputs with encoding targets.


In [ ]:
#  Helper functions — defined once, used throughout
def chronological_split(data, train_r=TRAIN_RATIO, val_r=VAL_RATIO):
    """Split a time-ordered DataFrame into train/val/test without shuffling."""
    n = len(data); n_tr = int(n*train_r); n_va = int(n*val_r)
    return data.iloc[:n_tr].copy(), data.iloc[n_tr:n_tr+n_va].copy(), data.iloc[n_tr+n_va:].copy()

def eval_forecast(true, pred, name='model'):
    """Return dict of forecasting metrics including directional accuracy."""
    true, pred = np.array(true).flatten(), np.array(pred).flatten()
    rmse = float(np.sqrt(mean_squared_error(true, pred)))
    mae  = float(mean_absolute_error(true, pred))
    mape = float(np.mean(np.abs((true-pred)/(true+1e-9)))*100)
    r2   = float(r2_score(true, pred))
    da   = float(np.mean((np.diff(true)>0)==(np.diff(pred)>0)))*100
    return dict(name=name, RMSE=round(rmse,2), MAE=round(mae,2),
                MAPE=round(mape,2), R2=round(r2,4), DirAcc=round(da,1))

def print_forecast_table(results):
    print(f"  {'Model':<30} {'RMSE':>7} {'MAE':>7} {'MAPE%':>7} {'R²':>7} {'Dir%':>7}")
    print("  "+"-"*64)
    best = min(results, key=lambda r: r['RMSE'])['name']
    for r in results:
        marker = " best" if r['name']==best else ""
        print(f"  {r['name']:<30} {r['RMSE']:>7.2f} {r['MAE']:>7.2f} "
              f"{r['MAPE']:>7.2f} {r['R2']:>7.4f} {r['DirAcc']:>7.1f}{marker}")

print("Helper functions defined ")

In [ ]:
# Chronological split
df_tr, df_va, df_te = chronological_split(df)
print(f"Train : {len(df_tr)} rows  ({df_tr['date'].min().date()} → {df_tr['date'].max().date()})")
print(f"Val   : {len(df_va)} rows  ({df_va['date'].min().date()} → {df_va['date'].max().date()})")
print(f"Test  : {len(df_te)} rows  ({df_te['date'].min().date()} → {df_te['date'].max().date()})")

#  Feature columns
CAT_COLS = ['market', 'commodity']
NUM_COLS = ['price_lag1','price_lag2','price_lag3',
            'rolling_mean_3','rolling_std_3','price_pct_change',
            'exchange_rate','producer_price_index',
            'month_sin','month_cos',
            'is_harvest_season','is_lean_season',
            'price_vs_annual','price_yoy']

# Fit ColumnTransformer on TRAIN ONLY
preprocessor = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
    ('num', 'passthrough', NUM_COLS),
])
preprocessor.fit(df_tr[CAT_COLS+NUM_COLS])

ohe_cols = preprocessor.named_transformers_['ohe'].get_feature_names_out(CAT_COLS).tolist()
FEATURES  = ohe_cols + NUM_COLS
print(f"\nPipeline fitted on training data only ")
print(f"OHE columns    : {ohe_cols}")
print(f"Total features : {len(FEATURES)}")

# Transform
X_tr = preprocessor.transform(df_tr[CAT_COLS+NUM_COLS])
X_va = preprocessor.transform(df_va[CAT_COLS+NUM_COLS])
X_te = preprocessor.transform(df_te[CAT_COLS+NUM_COLS])

# Encode target labels to integers (XGBoost requires this)
label_encoder = LabelEncoder()
y_tr = label_encoder.fit_transform(df_tr['decision'].values)  # fit on train only
y_va = label_encoder.transform(df_va['decision'].values)
y_te = label_encoder.transform(df_te['decision'].values)
class_names = label_encoder.classes_.tolist()

print(f"\nLabel encoding (fitted on training set only):")
for cn, enc in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {cn:<15} → {enc}")
print(f"\nX_train:{X_tr.shape}  X_val:{X_va.shape}  X_test:{X_te.shape}")

## Section 6 — Forecasting Pipeline

### Why chronological split is mandatory for time-series
Random splitting lets the model train on 2023 data and test on 2018 data —
it already "knows the future". We split strictly by date so the model
is always tested on genuinely unseen future data.

### Why the scaler is fitted on training data only
If we fit `MinMaxScaler` on the whole dataset, the scaler learns the global
min/max including future test prices. Training data is then normalised relative
to future price extremes — leaking future information into training.
Fix: fit on train only, transform val and test using those statistics.


In [ ]:
#  Aggregate to global time series
LSTM_FEAT_COLS = ['price','exchange_rate','producer_price_index',
                  'month_sin','month_cos',
                  'price_lag1','price_lag2','price_lag3',
                  'rolling_mean_3','rolling_std_3']
N_LSTM_FEAT = len(LSTM_FEAT_COLS)

ts_all = (df.groupby('date')[LSTM_FEAT_COLS]
            .mean().reset_index()
            .sort_values('date').dropna().reset_index(drop=True))

ts_tr, ts_va, ts_te = chronological_split(ts_all)
print(f"Time series: {len(ts_all)} months  |  Train:{len(ts_tr)} Val:{len(ts_va)} Test:{len(ts_te)}")

#  Fit scaler on TRAIN ONLY — critical to avoid leakage
lstm_scaler  = MinMaxScaler()
price_scaler = MinMaxScaler()
tr_sc = lstm_scaler.fit_transform(ts_tr[LSTM_FEAT_COLS].values)
va_sc = lstm_scaler.transform(ts_va[LSTM_FEAT_COLS].values)
te_sc = lstm_scaler.transform(ts_te[LSTM_FEAT_COLS].values)
price_scaler.fit(ts_tr[['price']].values)

#  Create multivariate sequences
def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len, :]); y.append(data[i+seq_len, 0:1])
    return np.array(X), np.array(y)

Xtr_l, ytr_l = make_sequences(tr_sc, SEQ_LEN)
Xva_l, yva_l = make_sequences(va_sc, SEQ_LEN)
Xte_l, yte_l = make_sequences(te_sc, SEQ_LEN)

def to_t(a): return torch.FloatTensor(a).to(DEVICE)
train_dl = DataLoader(TensorDataset(to_t(Xtr_l), to_t(ytr_l)), batch_size=LSTM_BATCH, shuffle=False)
val_dl   = DataLoader(TensorDataset(to_t(Xva_l), to_t(yva_l)), batch_size=LSTM_BATCH, shuffle=False)
print(f"Sequence shape: {Xtr_l.shape}  (samples × seq_len × features)")

### 6a — Multivariate LSTM with Early Stopping and Model Checkpointing

In [ ]:
class MultivariateLSTM(nn.Module):
    """
    Input  : (batch, SEQ_LEN, N_LSTM_FEAT) — 12 months × 10 features
    Output : (batch, 1) — next month's scaled price
    """
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, LSTM_HIDDEN, LSTM_LAYERS,
                            batch_first=True, dropout=LSTM_DROPOUT)
        self.fc   = nn.Linear(LSTM_HIDDEN, 1)

    def forward(self, x):
        h0 = torch.zeros(LSTM_LAYERS, x.size(0), LSTM_HIDDEN).to(DEVICE)
        c0 = torch.zeros(LSTM_LAYERS, x.size(0), LSTM_HIDDEN).to(DEVICE)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

lstm_model = MultivariateLSTM(N_LSTM_FEAT).to(DEVICE)
criterion  = nn.MSELoss()
optimizer  = torch.optim.Adam(lstm_model.parameters(), lr=LSTM_LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_val, patience_count = float('inf'), 0
train_losses, val_losses = [], []

print(f"Training LSTM (max {LSTM_EPOCHS} epochs, early stop patience={PATIENCE}) ...")
for epoch in range(LSTM_EPOCHS):
    lstm_model.train()
    ep_loss = 0
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(lstm_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    avg_tr = ep_loss / len(train_dl)

    lstm_model.eval()
    with torch.no_grad():
        avg_va = sum(criterion(lstm_model(xb), yb).item() for xb,yb in val_dl) / len(val_dl)

    train_losses.append(avg_tr); val_losses.append(avg_va)
    scheduler.step(avg_va)

    if avg_va < best_val:
        best_val, patience_count = avg_va, 0
        torch.save(lstm_model.state_dict(), CHECKPOINT)
    else:
        patience_count += 1

    if (epoch+1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}  train={avg_tr:.5f}  val={avg_va:.5f}  pat={patience_count}/{PATIENCE}")
    if patience_count >= PATIENCE:
        print(f"  Early stop at epoch {epoch+1}. Best val={best_val:.5f}")
        break

lstm_model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
print("Best LSTM weights restored ")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train loss', color='steelblue')
ax.plot(val_losses,   label='Val loss',   color='coral')
stop_ep = len(train_losses) - PATIENCE
if stop_ep > 0:
    ax.axvline(stop_ep, color='grey', linestyle='--', alpha=0.6, label='Early stop')
ax.set_title('LSTM Training & Validation Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss'); ax.legend()
plt.tight_layout(); plt.savefig('lstm_loss_curves.png', dpi=150); plt.show()

### 6b — Walk-Forward Validation

**Why walk-forward is better than a single train/test split for time-series:**

A single split evaluates the model on one fixed time window only. Walk-forward
validation simulates real deployment: the model is tested on the next window
after training on everything up to time *t*. This shows performance across
different market conditions — including both stable years and the volatile
2022-23 cedi collapse.

We report mean ± std of RMSE, MAE, MAPE, and Directional Accuracy across all folds.


In [ ]:
tscv_wf = TimeSeriesSplit(n_splits=5, test_size=max(SEQ_LEN+1, len(ts_all)//(5*2)))
wf_metrics = []
print(f"Walk-forward validation (5 folds) ...")

for fold, (tr_idx, te_idx) in enumerate(tscv_wf.split(ts_all[LSTM_FEAT_COLS].values)):
    if len(tr_idx) < SEQ_LEN + 5:
        continue
    fold_tr = ts_all[LSTM_FEAT_COLS].values[tr_idx]
    fold_te = ts_all[LSTM_FEAT_COLS].values[te_idx]

    sc2 = MinMaxScaler(); sc2.fit(fold_tr)
    combined = np.vstack([sc2.transform(fold_tr), sc2.transform(fold_te)])
    start = len(fold_tr)

    Xf, yf = [], []
    for i in range(start-SEQ_LEN, start-SEQ_LEN+len(fold_te)-1):
        if i < 0 or i+SEQ_LEN >= len(combined): continue
        Xf.append(combined[i:i+SEQ_LEN]); yf.append(combined[i+SEQ_LEN, 0])
    if len(Xf) == 0:
        continue

    lstm_model.eval()
    with torch.no_grad():
        pf_sc = lstm_model(to_t(np.array(Xf))).cpu().numpy().flatten()

    dummy  = np.zeros((len(pf_sc), N_LSTM_FEAT)); dummy[:,0]  = pf_sc
    dummy2 = np.zeros((len(yf),   N_LSTM_FEAT)); dummy2[:,0] = yf
    pf_real = sc2.inverse_transform(dummy)[:,0]
    yt_real = sc2.inverse_transform(dummy2)[:,0]

    m = eval_forecast(yt_real, pf_real, f'Fold {fold+1}')
    wf_metrics.append(m)
    print(f"  Fold {fold+1}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  DirAcc={m['DirAcc']:.1f}%")

if wf_metrics:
    print(f"\n  Walk-forward summary ({len(wf_metrics)} folds):")
    for metric in ['RMSE', 'MAE', 'MAPE', 'DirAcc']:
        vals = [m[metric] for m in wf_metrics]
        print(f"    {metric:<10}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

### 6c — Baseline Forecasting Models

**Why baselines are also tuned with `RandomizedSearchCV`:**

Comparing a tuned LSTM against an untuned Random Forest is not a fair
comparison. If RF uses `max_depth=6` and the optimal is `max_depth=12`,
we conclude the LSTM won when we just undertuned the RF.
Every model gets an equal opportunity to find its best configuration.

`Linear Regression` has no hyperparameters so it needs no tuning.
It is included as the simplest possible baseline — a straight-line fit.


In [ ]:
flat_sc = StandardScaler()
Xf_tr   = flat_sc.fit_transform(ts_tr[LSTM_FEAT_COLS].values[SEQ_LEN:])
Xf_te   = flat_sc.transform(ts_te[LSTM_FEAT_COLS].values[SEQ_LEN:])
yf_tr   = ts_tr['price'].values[SEQ_LEN:]
yf_te   = ts_te['price'].values[SEQ_LEN:]

# Linear Regression — no hyperparameters
lr_reg = LinearRegression()
lr_reg.fit(Xf_tr, yf_tr)
lr_preds = lr_reg.predict(Xf_te)
print("Linear Regression trained ")

# RF and XGBoost — tuned with RandomizedSearchCV + TimeSeriesSplit
tscv_reg = TimeSeriesSplit(n_splits=5)
reg_grids = {
    'RF Regressor': {
        'n_estimators':      randint(100, 400),
        'max_depth':         randint(3, 15),
        'min_samples_split': randint(2, 10),
        'min_samples_leaf':  randint(1, 5),
    },
    'XGBoost Regressor': {
        'n_estimators':  randint(100, 400),
        'max_depth':     randint(3, 7),
        'learning_rate': uniform(0.01, 0.2),
        'subsample':     uniform(0.6, 0.4),
    },
}
reg_bases = {
    'RF Regressor':      RandomForestRegressor(random_state=SEED, n_jobs=-1),
    'XGBoost Regressor': xgb.XGBRegressor(random_state=SEED),
}
bl_results = {}
print("\nTuning baseline regressors with RandomizedSearchCV ...")
print(f"  {'Model':<22} {'CV RMSE':>10} {'Test RMSE':>10}")
print("  "+"-"*44)
for name, base in reg_bases.items():
    Xr_tr = flat_sc.inverse_transform(Xf_tr)
    Xr_te = flat_sc.inverse_transform(Xf_te)
    search = RandomizedSearchCV(base, reg_grids[name], n_iter=30, cv=tscv_reg,
                                 scoring='neg_root_mean_squared_error',
                                 n_jobs=-1, random_state=SEED, refit=True)
    search.fit(Xr_tr, yf_tr)
    preds    = search.best_estimator_.predict(Xr_te)
    cv_rmse  = -search.best_score_
    te_rmse  = float(np.sqrt(mean_squared_error(yf_te, preds)))
    bl_results[name] = {'model':search.best_estimator_, 'preds':preds,
                         'metrics':eval_forecast(yf_te, preds, name), 'cv_rmse':cv_rmse}
    print(f"  {name:<22} {cv_rmse:>10.2f} {te_rmse:>10.2f}")

# Evaluate LSTM on test set
lstm_model.eval()
with torch.no_grad():
    lstm_preds_sc = lstm_model(to_t(Xte_l)).cpu().numpy()
lstm_preds = price_scaler.inverse_transform(lstm_preds_sc).flatten()
lstm_true  = price_scaler.inverse_transform(yte_l.reshape(-1,1)).flatten()
lstm_m     = eval_forecast(lstm_true, lstm_preds, 'LSTM (Global, Multivariate)')

all_fc = [lstm_m, eval_forecast(yf_te, lr_preds, 'Linear Regression')] +          [r['metrics'] for r in bl_results.values()]

print("\nFORECASTING MODEL COMPARISON:")
print_forecast_table(all_fc)
best_fc_name = min(all_fc, key=lambda r: r['RMSE'])['name']
print(f"\n🏆 Best forecasting model: {best_fc_name}")

# Plot
fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
fig.suptitle('Forecasting Model Comparison — Test Set', fontsize=13, fontweight='bold')

ax1 = fig.add_subplot(gs[0,:])
ax1.plot(lstm_true,  label='Actual',         color='#2c3e50', linewidth=2)
ax1.plot(lstm_preds, label='LSTM Predicted', color='steelblue', linewidth=1.8, linestyle='--')
ax1.set_title(f"LSTM — RMSE={lstm_m['RMSE']:.2f} GHS  R²={lstm_m['R2']:.3f}  DirAcc={lstm_m['DirAcc']:.1f}%")
ax1.set_ylabel('Price (GHS/100kg)'); ax1.legend()

for idx, (name, res) in enumerate(bl_results.items()):
    ax = fig.add_subplot(gs[1, idx])
    ax.plot(yf_te,        label='Actual',  color='#2c3e50', linewidth=1.5)
    ax.plot(res['preds'], label=name,      color='coral',   linewidth=1.5, linestyle='--')
    m = res['metrics']
    ax.set_title(f"{name}\nRMSE={m['RMSE']:.2f}  R²={m['R2']:.3f}", fontsize=9)
    ax.set_ylabel('Price (GHS)'); ax.legend(fontsize=7)

plt.savefig('forecasting_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

# Residuals
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
resid = lstm_true - lstm_preds
axes2[0].plot(resid, color='#8e44ad', alpha=0.8)
axes2[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes2[0].set_title('LSTM Residual Errors'); axes2[0].set_ylabel('Error (GHS)')
axes2[1].hist(resid, bins=25, color='#8e44ad', edgecolor='white', alpha=0.8)
axes2[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes2[1].set_title('Residual Distribution'); axes2[1].set_xlabel('Error (GHS)')
plt.tight_layout(); plt.savefig('lstm_residuals.png', dpi=150); plt.show()

## Section 7 — Classification Pipeline

### Why time-based splitting (not random)
Random splitting mixes records from 2023 into the training set and tests on 2018 data.
The model trains on the future. We split chronologically so the model is
always tested on the most recent genuinely unseen data.

### Class imbalance: class weights vs SMOTE
We compare both approaches and let the validation F1 decide.

**Class weights** (`class_weight='balanced'`): penalises the model more
heavily for misclassifying minority classes (STORE, SELL_PARTIAL).
No new data is created — all training samples are real market records.

**SMOTE**: generates synthetic minority-class samples by interpolating
between existing records. Risk for economic data: it creates imaginary market
states (e.g. a synthetic "Tamale-Maize-March-2021" price that never existed)
which can mislead the model. We test it anyway and compare.

### Tightened hyperparameter search space
The previous overfit gap was 0.55 (model memorised training data).
The fix is constraining the search space to force simpler models:
- `max_depth` capped at 4–6 (was 7–15)
- `min_samples_split` set high (was 2–10, now 10–30)
- `min_samples_leaf` set high (was 1–5, now 5–15)
- `min_child_weight` set high for XGBoost (was 1–6, now 5–20)

On a 1,500-row dataset, deep trees memorise perfectly. Shallow trees generalise.


In [ ]:
# Class weights vs SMOTE comparison
cw_array = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
cw_dict  = {int(k): round(v,4) for k,v in zip(np.unique(y_tr), cw_array)}
print("Class weights (balanced):", cw_dict)
print("Classes:", dict(zip(range(len(class_names)), class_names)))

# RF with class_weight
rf_cw = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                 max_depth=5, random_state=SEED, n_jobs=-1)
rf_cw.fit(X_tr, y_tr)
f1_cw_val = f1_score(y_va, rf_cw.predict(X_va), average='macro')
f1_cw_te  = f1_score(y_te, rf_cw.predict(X_te), average='macro')
print(f"\nRF class_weight='balanced':  Val F1={f1_cw_val:.4f}  Test F1={f1_cw_te:.4f}")

# RF with SMOTE
min_cc = pd.Series(y_tr).value_counts().min()
k_sm   = min(5, min_cc-1)
smote  = SMOTE(random_state=SEED, k_neighbors=k_sm)
X_sm, y_sm = smote.fit_resample(X_tr, y_tr)
rf_sm = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1)
rf_sm.fit(X_sm, y_sm)
f1_sm_val = f1_score(y_va, rf_sm.predict(X_va), average='macro')
f1_sm_te  = f1_score(y_te, rf_sm.predict(X_te), average='macro')
print(f"RF + SMOTE:                  Val F1={f1_sm_val:.4f}  Test F1={f1_sm_te:.4f}")

USE_SMOTE = (f1_sm_val > f1_cw_val)
print(f"\n→ Using {'SMOTE' if USE_SMOTE else 'class_weight=balanced'} (better validation F1)")
X_TRAIN_FINAL = X_sm if USE_SMOTE else X_tr
Y_TRAIN_FINAL = y_sm if USE_SMOTE else y_tr
assert Y_TRAIN_FINAL.dtype in [np.int32, np.int64, int, np.int_], "Labels must be integers for XGBoost"
print(f"Y dtype: {Y_TRAIN_FINAL.dtype}  unique: {np.unique(Y_TRAIN_FINAL)} ")

### 7a — Hyperparameter Tuning with constrained search spaces

In [ ]:
#  TIGHTENED search space — fixes the 0.55 overfit gap
# Deep trees (max_depth=15) memorise 1500-row training data perfectly.
# Shallow trees (max_depth=3-6) are forced to generalise.
# High min_samples_split/leaf also prevents memorisation.

CW = 'balanced' if not USE_SMOTE else None

param_grids = {
    'XGBoost': {
        'n_estimators':     randint(50, 150),
        'max_depth':        randint(2, 4),          # key constraint
        'learning_rate':    uniform(0.05, 0.15),
        'subsample':        uniform(0.5, 0.4),
        'min_child_weight': randint(5, 20),          # high = less overfit
        'colsample_bytree': uniform(0.5, 0.4),
    },
    'Random Forest': {
        'n_estimators':      randint(50, 150),
        'max_depth':         randint(3, 6),          # key constraint
        'min_samples_split': randint(10, 30),        # key constraint
        'min_samples_leaf':  randint(5, 15),         # key constraint
        'max_features':      ['sqrt'],
    },
    'Gradient Boosting': {
        'n_estimators':  randint(50, 150),
        'max_depth':     randint(2, 3),              # very shallow
        'learning_rate': uniform(0.01, 0.1),
        'subsample':     uniform(0.5, 0.4),
    },
    'Decision Tree': {
        'max_depth':         randint(2, 5),
        'min_samples_split': randint(15, 40),
        'min_samples_leaf':  randint(8, 20),
        'criterion':         ['gini', 'entropy'],
    },
    'Logistic Regression': {
        'C':       uniform(0.001, 1.0),              # strong regularisation
        'solver':  ['lbfgs', 'saga'],
        'max_iter':randint(500, 2000),
    },
}

base_models = {
    'XGBoost':             xgb.XGBClassifier(use_label_encoder=False,
                               eval_metric='mlogloss', random_state=SEED),
    'Random Forest':       RandomForestClassifier(class_weight=CW,
                               random_state=SEED, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(class_weight=CW, random_state=SEED),
    'Logistic Regression': LogisticRegression(class_weight=CW,
                               random_state=SEED, max_iter=1000),
}

tscv_cls  = TimeSeriesSplit(n_splits=5)
cls_results = {}

print("Running RandomizedSearchCV (n_iter=30, TimeSeriesSplit=5) ...")
print(f"  {'Model':<25} {'CV F1':>8} {'Val F1':>8} {'Test F1':>8} {'Tr F1':>8} {'Gap':>7} {'Flag':>6}")
print("  "+"="*70)

for name, base in base_models.items():
    search = RandomizedSearchCV(base, param_grids[name], n_iter=30, cv=tscv_cls,
                                 scoring='f1_macro', n_jobs=-1, random_state=SEED, refit=True)
    search.fit(X_TRAIN_FINAL, Y_TRAIN_FINAL)
    best   = search.best_estimator_
    f1_cv  = search.best_score_
    f1_va  = f1_score(y_va, best.predict(X_va), average='macro')
    f1_te  = f1_score(y_te, best.predict(X_te), average='macro')
    f1_tr  = f1_score(Y_TRAIN_FINAL, best.predict(X_TRAIN_FINAL), average='macro')
    gap    = f1_tr - f1_te
    flag   = "⚠️" if gap > 0.10 else "✅"
    cls_results[name] = dict(model=best, params=search.best_params_,
                              f1_cv=f1_cv, f1_val=f1_va, f1_test=f1_te,
                              f1_train=f1_tr, gap=gap, preds=best.predict(X_te))
    print(f"  {name:<25} {f1_cv:>8.4f} {f1_va:>8.4f} {f1_te:>8.4f} {f1_tr:>8.4f} {gap:>7.4f} {flag:>6}")

print("  "+"="*70)

## Section 8 — Explainability (SHAP)

SHAP (SHapley Additive exPlanations) shows **which features most influenced
each prediction** and in which direction.

For each decision class (STORE, SELL_NOW, SELL_PARTIAL) we see:
- Which features push the model toward that decision
- How much each feature contributes on average

This makes the model explainable to non-technical stakeholders — a field
extension officer can explain *why* the system says STORE rather than SELL_NOW.

**Explainer selection:**
- `XGBoost`, `Random Forest`, `Decision Tree` → `TreeExplainer` (fast, exact)
- `Logistic Regression` → `LinearExplainer`
- `GradientBoostingClassifier` → `KernelExplainer` (SHAP TreeExplainer does not
  support multiclass GBM — KernelExplainer is model-agnostic and works on everything)


In [ ]:
best_cls_name = max(cls_results, key=lambda k: cls_results[k]['f1_val'])
best_cls_res  = cls_results[best_cls_name]
best_cls      = best_cls_res['model']

print(f"Best classifier : {best_cls_name}")
print(f"Val  macro F1   : {best_cls_res['f1_val']:.4f}")
print(f"Test macro F1   : {best_cls_res['f1_test']:.4f}")
print(f"Overfit gap     : {best_cls_res['gap']:.4f}  "
      f"{' review needed' if best_cls_res['gap']>0.10 else ' clean'}")

print("\nComputing SHAP values ...")

def get_shap_values(model, X_bg, X_exp):
    """Select the correct SHAP explainer based on model type."""
    mtype = type(model).__name__
    if mtype == 'XGBClassifier':
        exp = shap.TreeExplainer(model); sv = exp.shap_values(X_exp)
        if isinstance(sv, list): return np.stack(sv, axis=2), 'TreeExplainer (XGBoost)'
        return (np.array(sv) if len(np.array(sv).shape)==3 else np.array(sv)), 'TreeExplainer (XGBoost)'
    elif mtype in ('RandomForestClassifier', 'DecisionTreeClassifier'):
        exp = shap.TreeExplainer(model); sv = exp.shap_values(X_exp)
        return (np.stack(sv, axis=2) if isinstance(sv, list) else np.array(sv)), 'TreeExplainer (RF/DT)'
    elif mtype == 'LogisticRegression':
        exp = shap.LinearExplainer(model, X_bg); sv = exp.shap_values(X_exp)
        return (np.stack(sv, axis=2) if isinstance(sv, list) else np.array(sv)), 'LinearExplainer'
    else:
        # GradientBoostingClassifier and anything else — KernelExplainer
        bg  = shap.sample(X_bg, min(50, len(X_bg)))
        exp = shap.KernelExplainer(model.predict_proba, bg)
        sv  = exp.shap_values(X_exp[:min(100, len(X_exp))], nsamples=100, silent=True)
        return (np.stack(sv, axis=2) if isinstance(sv, list) else np.array(sv)), f'KernelExplainer ({mtype})'

shap_array, explainer_used = get_shap_values(best_cls, X_tr, X_te)
print(f"{explainer_used} ")
print(f"Shape: {shap_array.shape}  (samples × features × classes)")

In [ ]:
#  SHAP plots — one bar chart per decision class
feat_labels = FEATURES

fig, axes = plt.subplots(1, len(class_names), figsize=(6*len(class_names), 6))
fig.suptitle('SHAP Feature Importance by Decision Class\n(mean |SHAP value| across test set)',
             fontsize=13, fontweight='bold')

for i, cls_name in enumerate(class_names):
    ax      = axes[i] if len(class_names) > 1 else axes
    sv_cls  = shap_array[:, :, i]
    mean_abs = np.abs(sv_cls).mean(axis=0)
    top_n    = min(12, len(mean_abs))
    top_idx  = np.argsort(mean_abs)[-top_n:]
    top_f    = [feat_labels[j] if j < len(feat_labels) else f'feat_{j}' for j in top_idx]
    color    = PALETTE.get(cls_name, '#4a90d9')
    ax.barh(top_f, mean_abs[top_idx], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Decision: {cls_name}', fontweight='bold', color=color)
    ax.set_xlabel('Mean |SHAP value|'); ax.tick_params(labelsize=8)

plt.tight_layout(); plt.savefig('shap_by_class.png', dpi=150, bbox_inches='tight'); plt.show()

# Overall importance across all classes
global_shap = np.abs(shap_array).mean(axis=(0, 2))
top_n2   = min(15, len(global_shap))
top_idx2 = np.argsort(global_shap)[-top_n2:]
top_f2   = [feat_labels[j] if j < len(feat_labels) else f'feat_{j}' for j in top_idx2]
fig2, ax2 = plt.subplots(figsize=(9, 6))
ax2.barh(top_f2, global_shap[top_idx2], color='#2980b9', edgecolor='white')
ax2.set_title(f'{best_cls_name} — Top {top_n2} Features (Overall)', fontweight='bold')
ax2.set_xlabel('Mean |SHAP value| across all classes')
plt.tight_layout(); plt.savefig('shap_overall.png', dpi=150, bbox_inches='tight'); plt.show()

# Plain-text summary
print("Top 5 features per decision class:")
for i, cls_name in enumerate(class_names):
    sv_cls  = shap_array[:, :, i]
    top5    = [feat_labels[j] if j < len(feat_labels) else f'feat_{j}'
               for j in np.argsort(np.abs(sv_cls).mean(axis=0))[-5:][::-1]]
    icon = {'STORE':'','SELL_PARTIAL':'','SELL_NOW':''}.get(cls_name,'')
    print(f"  {icon} {cls_name}: {', '.join(top5)}")

## Section 9 — Full Model Evaluation

In [ ]:
# ── Classification report
y_te_str   = label_encoder.inverse_transform(y_te)
preds_str  = label_encoder.inverse_transform(best_cls_res['preds'])

print(f"Classification Report — {best_cls_name} (test set, real-world proportions)")
print(classification_report(y_te_str, preds_str, target_names=class_names, digits=4))

#  Confusion matrix + per-class F1
cm = confusion_matrix(y_te_str, preds_str, labels=class_names)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Classifier Evaluation — {best_cls_name}', fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Test Set)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

per_cls = f1_score(y_te_str, preds_str, labels=class_names, average=None)
bc = [PALETTE.get(c,'steelblue') for c in class_names]
axes[1].bar(class_names, per_cls, color=bc, edgecolor='white')
axes[1].set_title('Per-Class F1 Score (Test Set)')
axes[1].set_ylabel('F1 Score'); axes[1].set_ylim(0, 1.1)
for i,(c,v) in enumerate(zip(class_names, per_cls)): axes[1].text(i, v+0.01, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

#  Full comparison table
print("\nFULL CLASSIFIER COMPARISON")
print(f"  {'Model':<25} {'CV F1':>8} {'Val F1':>8} {'Test F1':>8} {'Gap':>7} {'Flag':>6}")
print("  "+"="*65)
for name, res in cls_results.items():
    flag   = "⚠️" if res['gap'] > 0.10 else "✅"
    marker = " ← best" if name == best_cls_name else ""
    print(f"  {name:<25} {res['f1_cv']:>8.4f} {res['f1_val']:>8.4f} "
          f"{res['f1_test']:>8.4f} {res['gap']:>7.4f} {flag:>6}{marker}")
print("  "+"="*65)

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison Summary', fontsize=13, fontweight='bold')

names_cls = list(cls_results.keys()); x = np.arange(len(names_cls)); w = 0.35
axes[0].bar(x-w/2, [cls_results[n]['f1_val']  for n in names_cls], w, label='Val F1',  color='#2980b9')
axes[0].bar(x+w/2, [cls_results[n]['f1_test'] for n in names_cls], w, label='Test F1', color='#e67e22')
axes[0].set_title('Classifier — Val vs Test Macro F1')
axes[0].set_xticks(x); axes[0].set_xticklabels(names_cls, rotation=15, fontsize=8)
axes[0].set_ylim(0, 1.0); axes[0].legend()

gaps = [cls_results[n]['gap'] for n in names_cls]
gc   = ['#e74c3c' if g > 0.10 else '#27ae60' for g in gaps]
axes[1].bar(names_cls, gaps, color=gc, edgecolor='white')
axes[1].axhline(0.10, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Overfit Gap (Train − Test F1)\nRed line = 10% threshold')
axes[1].set_ylabel('Gap'); axes[1].set_xticks(range(len(names_cls)))
axes[1].set_xticklabels(names_cls, rotation=15, fontsize=8)

fc_n  = [r['name'] for r in all_fc]; fc_r = [r['RMSE'] for r in all_fc]
fc_c  = ['#f39c12' if n == best_fc_name else '#4a90d9' for n in fc_n]
axes[2].bar(fc_n, fc_r, color=fc_c, edgecolor='white')
axes[2].set_title('Forecasting — RMSE (lower = better)')
axes[2].set_ylabel('RMSE (GHS/100kg)'); axes[2].tick_params(axis='x', rotation=15, labelsize=8)

plt.tight_layout(); plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

## Section 10 — Conclusions

In [ ]:
print("="*65)
print("  POSTHARVEST IQ — FINAL RESULTS SUMMARY")
print("="*65)

lstm_final = next(r for r in all_fc if 'LSTM' in r['name'])

print(f"\n FORECASTING (Price Prediction)")
print(f"    Best model      : {best_fc_name}")
print(f"    LSTM RMSE       : GHS {lstm_final['RMSE']:.2f} per 100kg bag")
print(f"    LSTM MAE        : GHS {lstm_final['MAE']:.2f}")
print(f"    LSTM MAPE       : {lstm_final['MAPE']:.1f}%")
print(f"    LSTM R²         : {lstm_final['R2']:.4f}")
print(f"    Directional Acc : {lstm_final['DirAcc']:.1f}%  (% of months where price direction was correct)")

print(f"\n  CLASSIFICATION (Sell / Store Decision)")
print(f"    Best model      : {best_cls_name}")
print(f"    Val  macro F1   : {best_cls_res['f1_val']:.4f}")
print(f"    Test macro F1   : {best_cls_res['f1_test']:.4f}")
print(f"    Overfit gap     : {best_cls_res['gap']:.4f}  "
      f"({'⚠️ review needed' if best_cls_res['gap']>0.10 else '✅ clean'})")
print(f"    Balance method  : {'SMOTE' if USE_SMOTE else 'class_weight=balanced'}")

print(f"\n  BUSINESS INSIGHTS")
print( "    • Median net/bag = GHS -9.25. Storing is genuinely loss-making")
print( "      most months. SELL_NOW at 83% reflects real market economics.")
print( "    • Harvest months (Oct-Dec) are the most reliable STORE signal.")
print( "    • Exchange rate and producer price index transmit global shocks")
print( "      to local farm gate prices within 1-2 months.")
print( "    • The 2022-23 cedi collapse falls in the test set — the hardest")
print( "      period to predict, which partly explains performance drop.")
print( "    • price_vs_annual (current vs 12-month average) is a strong")
print( "      STORE signal: when prices are unusually high, consider storing.")

print(f"\n  LIMITATIONS")
print( "    • 1,549 rows is small for a neural network (LSTM needs more data).")
print( "    • Monthly data misses intra-month price spikes at harvest rush.")
print( "    • Storage cost is a national median — varies significantly by district.")
print( "    • Test period (2020-2023) includes COVID + cedi collapse —")
print( "      historically unusual years that are harder to predict.")

print(f"\n  FUTURE IMPROVEMENTS")
print( "    • Add rainfall/NDVI data for harvest-yield forecasting.")
print( "    • Collect more recent WFP price data to increase dataset size.")
print( "    • District-level models once more granular data is available.")
print( "    • Retrain monthly as new WFP data arrives (online learning).")
print( "    • Add confidence intervals on forecasts to communicate uncertainty.")
print("="*65)

## Section 11 — Save All Artifacts

| Artifact | Used by |
|---|---|
| `best_classifier.pkl` | `app/ml/predict.py` — decision classifier |
| `preprocessing_pipeline.pkl` | `app/ml/predict.py` — transforms new inputs the same way |
| `label_encoder.pkl` | `app/ml/predict.py` — decodes integer predictions to strings |
| `feature_columns.pkl` | `app/ml/predict.py` — correct feature order |
| `lstm_price_forecaster.pt` | `app/ml/predict.py` — LSTM weights |
| `lstm_scaler.pkl` | `app/ml/predict.py` — scales LSTM input |
| `price_scaler.pkl` | `app/ml/predict.py` — inverse-transforms LSTM output |
| `flat_scaler.pkl` | Baseline regressors |
| `model_metadata.json` | API dashboard endpoint and model info route |


In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(best_cls,      f'{MODELS_DIR}/best_classifier.pkl')
joblib.dump(preprocessor,  f'{MODELS_DIR}/preprocessing_pipeline.pkl')
joblib.dump(label_encoder, f'{MODELS_DIR}/label_encoder.pkl')
joblib.dump(FEATURES,      f'{MODELS_DIR}/feature_columns.pkl')
joblib.dump(lstm_scaler,   f'{MODELS_DIR}/lstm_scaler.pkl')
joblib.dump(price_scaler,  f'{MODELS_DIR}/price_scaler.pkl')
joblib.dump(flat_sc,       f'{MODELS_DIR}/flat_scaler.pkl')
shutil.copy(CHECKPOINT,    f'{MODELS_DIR}/lstm_price_forecaster.pt')

metadata = {
    'best_classifier':     best_cls_name,
    'best_params':         {k:str(v) for k,v in best_cls_res['params'].items()},
    'cv_macro_f1':         round(best_cls_res['f1_cv'],   4),
    'val_macro_f1':        round(best_cls_res['f1_val'],  4),
    'test_macro_f1':       round(best_cls_res['f1_test'], 4),
    'overfit_gap':         round(best_cls_res['gap'],     4),
    'overfit_detected':    best_cls_res['gap'] > 0.10,
    'balance_method':      'SMOTE' if USE_SMOTE else 'class_weight=balanced',
    'split_method':        'chronological_70_15_15_no_shuffle',
    'encoding':            'ColumnTransformer+OneHotEncoder (fitted on train only)',
    'target_encoding':     'LabelEncoder (fitted on train only)',
    'features':            FEATURES,
    'ohe_columns':         ohe_cols,
    'lstm_features':       LSTM_FEAT_COLS,
    'n_lstm_features':     N_LSTM_FEAT,
    'lstm_seq_len':        SEQ_LEN,
    'lstm_hidden':         LSTM_HIDDEN,
    'lstm_layers':         LSTM_LAYERS,
    'lstm_mae_ghs':        lstm_final['MAE'],
    'lstm_rmse_ghs':       lstm_final['RMSE'],
    'lstm_mape_pct':       lstm_final['MAPE'],
    'lstm_r2':             lstm_final['R2'],
    'lstm_dir_accuracy':   lstm_final['DirAcc'],
    'best_forecast_model': best_fc_name,
    'decision_classes':    class_names,
    'seed':                SEED,
    'all_cls_results': {
        k: {'cv_f1':round(v['f1_cv'],4),'val_f1':round(v['f1_val'],4),
            'test_f1':round(v['f1_test'],4),'gap':round(v['gap'],4)}
        for k,v in cls_results.items()
    },
    'all_forecast_results': {r['name']: r for r in all_fc},
}

with open(f'{MODELS_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Artifacts saved:")
for fname in sorted(os.listdir(MODELS_DIR)):
    if 'best.pt' not in fname:
        sz = os.path.getsize(f'{MODELS_DIR}/{fname}')
        print(f"  {fname:<48} {sz:>8,} bytes")
print("\n ALL ARTIFACTS SAVED")
print()
print(json.dumps({k:v for k,v in metadata.items()
                  if k not in ('all_cls_results','all_forecast_results',
                               'features','ohe_columns','lstm_features')}, indent=2))

## Section 12 — Download Everything

Copy the model files into `app/ml/models/` in VS Code after downloading.

In [ ]:
from google.colab import files

model_files = [
    f'{MODELS_DIR}/best_classifier.pkl',
    f'{MODELS_DIR}/preprocessing_pipeline.pkl',
    f'{MODELS_DIR}/label_encoder.pkl',
    f'{MODELS_DIR}/feature_columns.pkl',
    f'{MODELS_DIR}/lstm_price_forecaster.pt',
    f'{MODELS_DIR}/lstm_scaler.pkl',
    f'{MODELS_DIR}/price_scaler.pkl',
    f'{MODELS_DIR}/flat_scaler.pkl',
    f'{MODELS_DIR}/model_metadata.json',
]
chart_files = [
    'class_distribution.png', 'lstm_loss_curves.png',
    'forecasting_comparison.png', 'lstm_residuals.png',
    'confusion_matrix.png', 'model_comparison.png',
    'shap_by_class.png', 'shap_overall.png',
]

print("Downloading model files ...")
for fp in model_files:
    if os.path.exists(fp): files.download(fp); print(f"  ok {fp}")
    else: print(f"    {fp} not found — was Section 11 run?")

print("\nDownloading charts ...")
for fp in chart_files:
    if os.path.exists(fp): files.download(fp); print(f"  successful {fp}")
    else: print(f"  not successful  {fp} skipped")

print("\n Done.")
print("Next steps:")
print("  1. Copy model files into app/ml/models/ in VS Code")
print("  2. Run: uvicorn app.main:app --reload")
print("  3. Open http://localhost:8000/docs to test the API")
print("  4. git push to branch")